In [55]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [56]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Specify the exact file name within the dataset
file_path = "IMDB Dataset.csv"

# Use dataset_load to resolve the DeprecationWarning
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews",
  file_path,
)

print("First 5 records:\n", df.head())

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
First 5 records:
                                               review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [57]:
df.shape

(50000, 2)

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [59]:
df["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [60]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [61]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((40000,), (10000,), (40000,), (10000,))

In [62]:
y_train.head()

,sentiment
39087,negative
30893,negative
45278,positive
16398,negative
13653,negative


In [63]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

In [64]:
y_train = le.fit_transform(y_train)
y_test = le.fit(y_test)

y_train, y_test

(array([0, 0, 1, ..., 0, 1, 1]), LabelEncoder())

In [65]:
le.classes_, le.transform(le.classes_)

(array(['negative', 'positive'], dtype=object), array([0, 1]))

In [66]:
import joblib
joblib.dump(le, 'le.joblib')

['le.joblib']

In [67]:
word_counts = X_train.str.split().str.len()

word_counts.describe()

,review
count,40000.000000
mean,231.006425
std,171.612227
min,4.000000
25%,126.000000
50%,173.000000
75%,280.000000
max,2470.000000


In [68]:
word_counts.quantile(0.90)

np.float64(450.0)

In [69]:
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_features = 10000
max_sequence_len = 450

X_train= [one_hot(text, n=max_features) for text in X_train]
X_test = [one_hot(text, n=max_features) for text in X_test]


X_train = pad_sequences(
    X_train,
    maxlen=max_sequence_len,
    padding='pre',
    truncating='pre'
)

X_test = pad_sequences(
    X_test,
    maxlen=max_sequence_len,
    padding="pre",
    truncating="pre"
)

In [70]:
np.savez_compressed('X_data.npz', X_train=X_train, X_test=X_test)
np.savez_compressed('y_data.npz', y_train=y_train, y_test=y_test)